In [98]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [99]:
OPENROUTER_KEY = os.getenv("OPENROUTER_KEY")
if not OPENROUTER_KEY:
    raise ValueError("OPENROUTER_KEY is missing. Please add it to your .env file.")

In [ ]:
from langchain_openrouter import ChatOpenRouter

model_id = "dots-studio/dots-3-note-preview:free"

model = ChatOpenRouter(
    model=model_id,
    temperature=0.8,
    api_key=OPENROUTER_KEY
)

print("model:", model.model_name)

model: dots-studio/dots-3-note-preview:free


In [101]:
# ============================================================
# Load Date: Configure PDF Directory
# ============================================================

from pathlib import Path

DATA_DIR = Path("data")
pdf_files = sorted(DATA_DIR.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files:")
for pdf in pdf_files:
    print(f" - {pdf.name}")

Found 5 PDF files:
 - 01_Employee_Handbook.pdf
 - 02_Organization_Structure.pdf
 - 03_Employee_Directory.pdf
 - 04_Project_Portfolio.pdf
 - 05_Client_Profiles.pdf


In [102]:
# ============================================================
# Load PDF Documents
# ============================================================

from langchain_community.document_loaders import PyPDFLoader

documents = []

for pdf_path in pdf_files:
    print(f"Loading: {pdf_path.name}")

    loader = PyPDFLoader(str(pdf_path))
    documents.extend(loader.load())

print("\n" + "=" * 60)
print(f"Total PDF Files : {len(pdf_files)}")
print(f"Total Pages     : {len(documents)}")
print("=" * 60)

Loading: 01_Employee_Handbook.pdf
Loading: 02_Organization_Structure.pdf
Loading: 03_Employee_Directory.pdf
Loading: 04_Project_Portfolio.pdf
Loading: 05_Client_Profiles.pdf

Total PDF Files : 5
Total Pages     : 98


In [103]:

# ============================================================
# Preview Loaded Documents
# ============================================================

print(f"Source : {documents[0].metadata['source']}")
print(f"Page   : {documents[0].metadata['page']}")
print("-" * 60)
print(documents[0].page_content[:1000])

Source : data\01_Employee_Handbook.pdf
Page   : 0
------------------------------------------------------------
NEXATECH SOLUTIONS
Cloud · AI · Cyber Security · Data Engineering
Employee Handbook
Policies, Benefits & Ways of Working
Document Number NTS-HR-001
Version 6.2
Effective Date January 15, 2026
Document Owner Grace Okafor, Director of People Operations 
(Human Resources)
Classification Internal Use Only
NexaTech Solutions — Internal Use Only   |   Page 1 of 8


In [104]:

# ============================================================
# Split Documents into Chunks
# ============================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

split_documents = text_splitter.split_documents(documents)

print("=" * 60)
print(f"Original Documents : {len(documents)}")
print(f"Total Chunks       : {len(split_documents)}")
print("=" * 60)

Original Documents : 98
Total Chunks       : 158


In [105]:
# Preview one chunk instead of printing all 158 documents.
print(split_documents[0].page_content[:1000])
print(split_documents[0].metadata)

NEXATECH SOLUTIONS
Cloud · AI · Cyber Security · Data Engineering
Employee Handbook
Policies, Benefits & Ways of Working
Document Number NTS-HR-001
Version 6.2
Effective Date January 15, 2026
Document Owner Grace Okafor, Director of People Operations 
(Human Resources)
Classification Internal Use Only
NexaTech Solutions — Internal Use Only   |   Page 1 of 8
{'producer': 'LibreOffice 24.2', 'creator': 'Writer', 'creationdate': '2026-07-30T03:34:56+00:00', 'author': 'Un-named', 'source': 'data\\01_Employee_Handbook.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}


In [106]:
# ============================================================
# Convert Documents into Graph Documents
# ============================================================

from langchain_experimental.graph_transformers import LLMGraphTransformer

# GPT-OSS on Groq can return malformed native tool-call JSON.
# Use the transformer's JSON parsing path instead of Groq tool calling.
llm_transformer = LLMGraphTransformer(
    llm=model,
)

graph_documents = llm_transformer.convert_to_graph_documents(
    split_documents[:20]
)

print("=" * 60)
print(f"Graph Documents Created : {len(graph_documents)}")
print("=" * 60)

Graph Documents Created : 20


In [109]:
# ============================================================
# Preview Extracted Graph
# ============================================================

graph = graph_documents[0]

print("=" * 60)
print("NODES")
print("=" * 60)

for node in graph.nodes:
    print(node)

print("\n" + "=" * 60)
print("RELATIONSHIPS")
print("=" * 60)

for relationship in graph.relationships:
    print(relationship)

NODES
id='Nexatech Solutions' type='Organization' properties={}
id='Employee Handbook' type='Document' properties={}
id='Grace Okafor' type='Person' properties={}
id='Cloud' type='Concept' properties={}
id='Ai' type='Concept' properties={}
id='Cyber Security' type='Concept' properties={}
id='Data Engineering' type='Concept' properties={}
id='Director Of People Operations' type='Role' properties={}
id='Human Resources' type='Department' properties={}
id='Nts-Hr-001' type='Identifier' properties={}
id='Version 6.2' type='Version' properties={}
id='January 15, 2026' type='Date' properties={}
id='Internal Use Only' type='Classification' properties={}

RELATIONSHIPS
source=Node(id='Nexatech Solutions', type='Organization', properties={}) target=Node(id='Employee Handbook', type='Document', properties={}) type='HAS_DOCUMENT' properties={}
source=Node(id='Employee Handbook', type='Document', properties={}) target=Node(id='Grace Okafor', type='Person', properties={}) type='HAS_OWNER' propertie

In [112]:
# ============================================================
# Preview Multiple Graph Documents
# ============================================================

for index, graph in enumerate(graph_documents[:3], start=1):

    print("\n" + "=" * 70)
    print(f"Graph Document {index}")
    print("=" * 70)

    print(f"Nodes         : {len(graph.nodes)}")
    print(f"Relationships : {len(graph.relationships)}")


Graph Document 1
Nodes         : 13
Relationships : 12

Graph Document 2
Nodes         : 4
Relationships : 3

Graph Document 3
Nodes         : 30
Relationships : 28


In [119]:
# ============================================================
# Connect to Neo4j
# ============================================================

from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "GraphRAG@123"

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

print("✅ Connected to Neo4j")

✅ Connected to Neo4j


In [120]:
def sanitize(text: str) -> str:
    return (
        text.upper()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace(".", "_")
    )


def store_graph_documents(driver, graph_documents):
    with driver.session() as session:
        for graph_document in graph_documents:

            # --------------------------------------------------
            # Store Nodes
            # --------------------------------------------------

            for node in graph_document.nodes:
                label = sanitize(node.type)
                properties = {"id": node.id, "name": node.id}

                if hasattr(node, "properties"):
                    properties.update(node.properties)

                query = f"""
                        MERGE (n:Entity:{label} {{id: $id}})
                        SET n += $properties
                        """
                session.run(query, id=node.id, properties=properties)

            # --------------------------------------------------
            # Store Relationships
            # --------------------------------------------------

            for relationship in graph_document.relationships:
                source_label = sanitize(relationship.source.type)
                target_label = sanitize(relationship.target.type)
                relationship_type = sanitize(relationship.type)
                properties = {}

                if hasattr(relationship, "properties"):
                    properties.update(relationship.properties)

                query = (
                    f"MATCH (a:{source_label} {{id: $source}}) "
                    f"MATCH (b:{target_label} {{id: $target}}) "
                    f"MERGE (a)-[r:{relationship_type}]->(b) "
                    f"SET r += $properties"
                )

                session.run(
                    query,
                    source=relationship.source.id,
                    target=relationship.target.id,
                    properties=properties,
                )

    print("✅ Graph stored successfully.")

In [121]:
# ============================================================
# Store Graph into Neo4j
# ============================================================

store_graph_documents(
    driver=driver,
    graph_documents=graph_documents
)

print("✅ Graph successfully ingested into Neo4j.")

✅ Graph stored successfully.
✅ Graph successfully ingested into Neo4j.


In [ ]:
# ============================================================
# User Question
# ============================================================
question = "Tell me about Cyber Security"

In [123]:
# ============================================================
# Retrieve Matching Graph Context
# ============================================================

from neo4j import GraphDatabase

query = """
MATCH (n)-[r]-(m)

WHERE toLower($question) CONTAINS toLower(n.name)
   OR toLower(n.name) CONTAINS toLower($question)

RETURN
    labels(n) AS source_type,
    n.name AS source,
    type(r) AS relationship,
    labels(m) AS target_type,
    m.name AS target

LIMIT 50
"""

with driver.session() as session:

    result = session.run(
        query,
        question=question
    )

    graph_context = []

    for record in result:

        graph_context.append(
            f"{record['source']} "
            f"-[{record['relationship']}]-> "
            f"{record['target']}"
        )

print("\n".join(graph_context))

Cyber Security -[COVERS]-> Employee Handbook
Cyber Security -[PROVIDES_SERVICE]-> Nexatech Solutions
Cyber Security -[HAS_DEPARTMENT]-> Nexatech Solutions


In [124]:
# ============================================================
# Generate Final Answer
# ============================================================

context = "\n".join(graph_context)

prompt = f"""
You are a helpful assistant.

Answer ONLY using the graph context.

Graph Context:
{context}

Question:
{question}
"""

response = model.invoke(prompt)

print(response.content)



Based on the graph context, here is what is known about Cyber Security:

*   It **covers** the Employee Handbook.
*   It **provides a service** to Nexatech Solutions.
*   It is a **department** within Nexatech Solutions.


In [ ]:
# ============================================================
# Extract Entities from User Question
# ============================================================

from pydantic import BaseModel, Field
from typing import List


class EntityExtraction(BaseModel):
    entities: List[str] = Field(
        description="List of important entities mentioned in the question."
    )


entity_llm = model.with_structured_output(EntityExtraction)

question = "Tell me about Cyber Security"

entity_result = entity_llm.invoke(f"""
Extract all important entities from the following question.

Question:
{question}
""")

print("=" * 60)
print("Extracted Entities")
print("=" * 60)

print(entity_result.entities)

Extracted Entities
['Cyber Security']


In [127]:
entity_result

EntityExtraction(entities=['Cyber Security'])

In [131]:
# ============================================================
# Retrieve Relevant Graph Context
# ============================================================
cypher_query = """
MATCH path = (n)-[*1..2]-(m)

WHERE toLower(n.name) = toLower($entity)

UNWIND relationships(path) AS r
UNWIND nodes(path) AS node

RETURN DISTINCT
    startNode(r).name AS source,
    type(r) AS relationship,
    endNode(r).name AS target
LIMIT 100
"""

graph_context = []

with driver.session() as session:

    for entity in entity_result.entities:

        result = session.run(
            cypher_query,
            entity=entity
        )

        for record in result:

            graph_context.append(
                f"{record['source']} "
                f"-[{record['relationship']}]-> "
                f"{record['target']}"
            )

print("=" * 60)
print("Retrieved Graph Context")
print("=" * 60)

print("\n".join(graph_context))

Retrieved Graph Context
Employee Handbook -[COVERS]-> Cyber Security
Nexatech Solutions -[HAS_DOCUMENT]-> Employee Handbook
Employee Handbook -[HAS_OWNER]-> Grace Okafor
Employee Handbook -[COVERS]-> Cloud
Employee Handbook -[COVERS]-> Ai
Employee Handbook -[COVERS]-> Data Engineering
Employee Handbook -[HAS_NUMBER]-> Nts-Hr-001
Employee Handbook -[HAS_VERSION]-> Version 6.2
Employee Handbook -[EFFECTIVE_DATE]-> January 15, 2026
Employee Handbook -[CLASSIFICATION]-> Internal Use Only
Nexatech Solutions -[CREATOR]-> Employee Handbook
Employee Handbook -[HAS_ID]-> Nts-Hr-001
Employee Handbook -[HAS_RESTRICTION]-> Internal Use Only
Nexatech Solutions -[PUBLISHES]-> Employee Handbook
Employee Handbook -[SPECIFIES]-> Voluntary Resignation
Employee Handbook -[SPECIFIES]-> Involuntary Separation
Employee Handbook -[SPECIFIES]-> Final Pay
Employee Handbook -[SPECIFIES]-> Exit Interview
Employee Handbook -[SPECIFIES]-> It Assets Return
Employee Handbook -[SPECIFIES]-> Client Systems Access Revo

In [132]:
# ============================================================
# Generate Final Answer
# ============================================================

context = "\n".join(graph_context)

prompt = f"""
You are an AI assistant.

Answer ONLY using the graph context.

If the answer is not available, say so.

==========================
Graph Context
==========================

{context}

==========================
Question
==========================

{question}
"""

response = model.invoke(prompt)

print("=" * 60)
print("Final Answer")
print("=" * 60)

print(response.content)

Final Answer


Based on the graph context, here is the information about Cyber Security:

*   **Service Provider:** Nexatech Solutions provides Cyber Security as a service.
*   **Department:** Nexatech Solutions has a dedicated Cyber Security department.
*   **Sub-department:** This department includes a Security Operations Center.
*   **Documentation:** The Employee Handbook covers policies and procedures related to Cyber Security.
*   **Related Policies:** The company's IT Asset Policy is also relevant to Cyber Security practices.


In [133]:
# ============================================================
# Retrieve Relevant Graph Context from Neo4j
# ============================================================

cypher_query = """
MATCH (n)-[r]-(m)

WHERE ANY(keyword IN $keywords
          WHERE toLower(n.name) CONTAINS keyword)

RETURN DISTINCT
    labels(n) AS source_label,
    n.name AS source,
    type(r) AS relationship,
    labels(m) AS target_label,
    m.name AS target

LIMIT 100
"""

graph_context = []

with driver.session() as session:

    for entity in entity_result.entities:

        # Split extracted entity into keywords
        keywords = [
            word.strip()
            for word in entity.lower().split()
            if len(word.strip()) > 2
        ]

        print(f"\nSearching for Entity : {entity}")
        print(f"Keywords             : {keywords}")

        result = session.run(
            cypher_query,
            keywords=keywords
        )

        records = list(result)

        print(f"Matches Found        : {len(records)}")

        for record in records:

            graph_context.append(
                f"{record['source']} "
                f"-[{record['relationship']}]-> "
                f"{record['target']}"
            )

print("\n" + "=" * 60)
print("Retrieved Graph Context")
print("=" * 60)

print("\n".join(graph_context))


Searching for Entity : Cyber Security
Keywords             : ['cyber', 'security']
Matches Found        : 15

Retrieved Graph Context
Cyber Security -[COVERS]-> Employee Handbook
Cyber Security -[PROVIDES_SERVICE]-> Nexatech Solutions
Security Operations Center -[MAINTAINS]-> 24X7 Monitoring Coverage
Security Operations Center -[LED_FROM]-> Bangalore
Security Operations Center -[LED_FROM]-> London
Security-Incidents@Nexatechsolutions.Com -[REPORTED_TO]-> Security Incidents
Cyber Security Team -[DEFINED_BY]-> Restricted Data Handling Standard
Security Incidents -[REPORTED_TO]-> Nexahelp It Service Desk
Security Incidents -[REPORTED_TO]-> Security-Incidents@Nexatechsolutions.Com
Cyber Security Security Operations Center -[MAINTAINS]-> 24X7 Monitoring Coverage
Cyber Security Security Operations Center -[LOCATED_IN]-> Bangalore
Cyber Security Security Operations Center -[LOCATED_IN]-> London
Cyber Security Security Operations Center -[HAS_DEPARTMENT]-> Nexatech Solutions
Cyber Security Te

In [134]:
question

'Tell me about Cyber Security'

In [136]:
# ============================================================
# Generate Final Answer
# ============================================================

context = "\n".join(graph_context)

prompt = f"""
You are an AI assistant.

Answer ONLY using the graph context.

If the answer is not available, say so.

==========================
Graph Context
==========================

{context}

==========================
Question
==========================

{question}
"""

response = model.invoke(prompt)

print("=" * 60)
print("Final Answer")
print("=" * 60)

print(response.content)

Final Answer


Based on the graph context, here is a summary of information about Cyber Security:

*   **Coverage & Service:** Cyber Security covers the Employee Handbook and provides its service to Nexatech Solutions.
*   **Operations:** It maintains a 24x7 Monitoring Coverage through the Security Operations Center (SOC). This SOC is located in both Bangalore and London.
*   **Team & Standards:** The Cyber Security Team is defined by the Restricted Data Handling Standard.
*   **Incident Response:** Security incidents are reported to the Nexahelp IT Service Desk and to the email address Security-Incidents@Nexatechsolutions.Com. The team also performs actions like Client Systems Access Revocation.


In [137]:
# ============================================================
# Create Full-Text Index (If Not Exists)
# ============================================================

CHECK_INDEX_QUERY = """
SHOW INDEXES
YIELD name
WHERE name = 'entity_index'
RETURN count(*) AS count
"""

CREATE_INDEX_QUERY = """
CREATE FULLTEXT INDEX entity_index
FOR (n:Entity)
ON EACH [n.name]
"""

with driver.session() as session:

    result = session.run(CHECK_INDEX_QUERY).single()

    if result["count"] == 0:

        session.run(CREATE_INDEX_QUERY)

        print("✅ Full-Text Index created successfully.")

    else:

        print("ℹ️ Full-Text Index already exists.")

✅ Full-Text Index created successfully.


In [138]:
# ============================================================
# User Question
# ============================================================

question = "Tell me about Cyber Security"

# ============================================================
# Extract Entities
# ============================================================

from pydantic import BaseModel, Field
from typing import List


class EntityExtraction(BaseModel):
    entities: List[str] = Field(
        description="List of entities mentioned in the user question."
    )


entity_llm = model.with_structured_output(EntityExtraction)

entity_result = entity_llm.invoke(f"""
Extract all important entities from the question.

Question:
{question}
""")

print("=" * 60)
print("Extracted Entities")
print("=" * 60)

print(entity_result.entities)


# ============================================================
# Full-Text Search
# ============================================================

FULLTEXT_QUERY = """
CALL db.index.fulltext.queryNodes(
    'entity_index',
    $entity
)
YIELD node, score

RETURN
    node.name AS name,
    labels(node) AS labels,
    score

ORDER BY score DESC
LIMIT 5
"""

matched_entities = []

with driver.session() as session:

    for entity in entity_result.entities:

        print(f"\nSearching : {entity}")

        result = session.run(
            FULLTEXT_QUERY,
            entity=entity
        )

        records = list(result)

        if not records:
            print("No match found.")
            continue

        for record in records:

            print(
                f"{record['name']} "
                f"(Score : {record['score']:.2f})"
            )

            matched_entities.append(record["name"])

Extracted Entities
['Cyber Security']

Searching : Cyber Security
Cyber Security (Score : 3.96)
Cyber Security (Score : 3.96)
Cyber Security (Score : 3.96)
Cyber Security Team (Score : 3.36)
Cyber Security Team (Score : 3.36)


In [139]:
# ============================================================
# Retrieve Graph Context for Matched Entities
# ============================================================

GRAPH_QUERY = """
MATCH (n)-[r]-(m)
WHERE n.name IN $entities

RETURN DISTINCT
    n.name AS source,
    type(r) AS relationship,
    m.name AS target
LIMIT 100
"""

graph_context = []

with driver.session() as session:

    result = session.run(
        GRAPH_QUERY,
        entities=matched_entities
    )

    for record in result:

        graph_context.append(
            f"{record['source']} "
            f"-[{record['relationship']}]-> "
            f"{record['target']}"
        )

print("=" * 60)
print("Retrieved Graph Context")
print("=" * 60)

for item in graph_context:
    print(item)

Retrieved Graph Context
Cyber Security -[COVERS]-> Employee Handbook
Cyber Security -[PROVIDES_SERVICE]-> Nexatech Solutions
Cyber Security Team -[DEFINED_BY]-> Restricted Data Handling Standard
Cyber Security Team -[PERFORMED_BY]-> Client Systems Access Revocation
Cyber Security -[HAS_DEPARTMENT]-> Nexatech Solutions


In [141]:
# ============================================================
# Generate Final Answer
# ============================================================

context = "\n".join(graph_context)

prompt = f"""
You are a GraphRAG assistant.

Answer the user's question using only the information
provided in the graph context.

If the graph context does not contain enough information,
say that the information is not available in the knowledge graph.

-------------------- Graph Context --------------------

{context}

---------------------- Question ------------------------

{question}

---------------------------------------------------------

Provide a concise and factual answer.
"""

response = model.invoke(prompt)

print("=" * 60)
print("Final Answer")
print("=" * 60)

print(response.content)

Final Answer


Based on the graph context, here is a concise summary of Cyber Security:

*   It **covers** the Employee Handbook.
*   It **provides service** to Nexatech Solutions.
*   Nexatech Solutions is listed as a **department** of Cyber Security.
*   The associated Cyber Security Team is **defined by** the Restricted Data Handling Standard.
*   The Cyber Security Team **performs** the action of Client Systems Access Revocation.
